
# AgroUTMACH PIX
## NB03 — Exploración del video, detección y conteo de aves visibles

En esta etapa utilizaremos el modelo entrenado en NB02 para:

1. inspeccionar técnicamente `0002.mp4`;
2. muestrear automáticamente escenas a lo largo del video;
3. detectar pollos con YOLO11s;
4. identificar escenas de alta, media y baja densidad;
5. generar clips candidatos para la feria;
6. procesar un clip de alta densidad con detección y conteo por fotograma;
7. guardar un CSV con el número de **aves visibles/detectadas**.

> El conteo de este notebook es por fotograma. No representa todavía individuos únicos del lote. El seguimiento temporal se incorporará en NB04.

**Entorno recomendado: GPU NVIDIA L4.**


## 0. Active GPU antes de ejecutar

**Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. Si aparece **L4**, selecciónela.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi
!pip -q install -U ultralytics

In [ ]:

from pathlib import Path
import os, shutil, json, time, math
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

ROOT = Path('/content/drive/MyDrive/AgroUTMACH_PIX')
VIDEO_DRIVE = ROOT / '00_ZENODO_ORIGINAL' / '0002.mp4'
MODEL_DRIVE = ROOT / '03_MODELOS' / 'AgroUTMACH_PIX_YOLO11s_PIO_best.pt'
VIDEOS_OUT = ROOT / '04_VIDEOS_FERIA'
RESULTS_OUT = ROOT / '05_RESULTADOS' / 'NB03_VIDEO'
VIDEOS_OUT.mkdir(parents=True, exist_ok=True)
RESULTS_OUT.mkdir(parents=True, exist_ok=True)
assert VIDEO_DRIVE.exists(), f'No se encontró {VIDEO_DRIVE}'
assert MODEL_DRIVE.exists(), f'No se encontró {MODEL_DRIVE}'
assert torch.cuda.is_available(), 'No hay GPU activa.'
print('GPU:', torch.cuda.get_device_name(0))
print('Video:', VIDEO_DRIVE)
print('Modelo:', MODEL_DRIVE)


## 1. Copiar el video al disco local de Colab

In [ ]:

VIDEO_LOCAL = Path('/content/0002.mp4')
if not VIDEO_LOCAL.exists() or VIDEO_LOCAL.stat().st_size != VIDEO_DRIVE.stat().st_size:
    print('Copiando video a /content ...')
    t0=time.time(); shutil.copy2(VIDEO_DRIVE, VIDEO_LOCAL)
    print(f'✓ Copia completada en {(time.time()-t0)/60:.1f} min')
else:
    print('↪ El video local ya existe.')
print(f'Tamaño: {VIDEO_LOCAL.stat().st_size/1024**3:.2f} GB')


## 2. Leer metadatos del video

In [ ]:

cap=cv2.VideoCapture(str(VIDEO_LOCAL)); assert cap.isOpened(), 'No se pudo abrir el video.'
fps=cap.get(cv2.CAP_PROP_FPS); n_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration=n_frames/fps if fps>0 else 0; cap.release()
print(f'Resolución: {width} x {height}')
print(f'FPS: {fps:.3f}')
print(f'Fotogramas: {n_frames:,}')
print(f'Duración: {duration/60:.2f} min')
metadata={'video':VIDEO_DRIVE.name,'width':width,'height':height,'fps':fps,'frames':n_frames,'duration_seconds':duration}
with open(RESULTS_OUT/'video_metadata.json','w',encoding='utf-8') as f: json.dump(metadata,f,indent=2,ensure_ascii=False)


## 3. Explorar 12 momentos del video con el modelo entrenado

In [ ]:

model=YOLO(str(MODEL_DRIVE))
CONF=0.25; IOU=0.60; IMGSZ=960; MAX_DET=1000; N_SAMPLES=12
sample_times=np.linspace(duration*0.05,duration*0.95,N_SAMPLES)
records=[]; annotated_samples=[]
cap=cv2.VideoCapture(str(VIDEO_LOCAL))
for i,tsec in enumerate(sample_times,start=1):
    cap.set(cv2.CAP_PROP_POS_MSEC,float(tsec*1000)); ok,frame=cap.read()
    if not ok: continue
    result=model.predict(source=frame,imgsz=IMGSZ,conf=CONF,iou=IOU,max_det=MAX_DET,device=0,verbose=False)[0]
    n=len(result.boxes) if result.boxes is not None else 0
    mean_conf=float(result.boxes.conf.mean().item()) if n>0 else 0.0
    annotated=cv2.cvtColor(result.plot(labels=False,conf=False),cv2.COLOR_BGR2RGB)
    records.append({'sample':i,'time_seconds':float(tsec),'time_minutes':float(tsec/60),'visible_birds':int(n),'mean_confidence':mean_conf})
    annotated_samples.append((tsec,n,annotated))
    print(f'[{i:02d}/{N_SAMPLES}] {tsec/60:6.2f} min | aves visibles={n:4d} | conf={mean_conf:.3f}')
cap.release()
df_samples=pd.DataFrame(records); df_samples.to_csv(RESULTS_OUT/'NB03_muestreo_video.csv',index=False)
df_samples


## 4. Crear mosaico de escenas candidatas

In [ ]:

cols=3; rows=math.ceil(len(annotated_samples)/cols)
fig,axes=plt.subplots(rows,cols,figsize=(18,5.5*rows)); axes=np.array(axes).reshape(-1)
for ax in axes: ax.axis('off')
for ax,(tsec,n,img) in zip(axes,annotated_samples):
    ax.imshow(img); ax.set_title(f'{tsec/60:.1f} min | aves visibles: {n}',fontsize=11); ax.axis('off')
plt.tight_layout(); montage_path=RESULTS_OUT/'NB03_mosaico_12_escenas.png'
plt.savefig(montage_path,dpi=180,bbox_inches='tight'); plt.show(); print('✓',montage_path)


## 5. Seleccionar escenas de alta, media y baja densidad

In [ ]:

assert len(df_samples)>=3
idx_high=df_samples['visible_birds'].idxmax(); idx_low=df_samples['visible_birds'].idxmin(); med=df_samples['visible_birds'].median(); idx_mid=(df_samples['visible_birds']-med).abs().idxmin()
candidates={'alta':df_samples.loc[idx_high].to_dict(),'media':df_samples.loc[idx_mid].to_dict(),'baja':df_samples.loc[idx_low].to_dict()}
print(json.dumps(candidates,indent=2,ensure_ascii=False))
with open(RESULTS_OUT/'NB03_candidatos_segmentos.json','w',encoding='utf-8') as f: json.dump(candidates,f,indent=2,ensure_ascii=False)


## 6. Extraer tres clips originales de 45 segundos

In [ ]:

CLIP_SECONDS=45
def clip_start(center_sec):
    start=center_sec-CLIP_SECONDS/2
    return max(0.0,min(start,max(0.0,duration-CLIP_SECONDS)))
clip_paths={}
for label,info in candidates.items():
    start=clip_start(float(info['time_seconds']))
    out=VIDEOS_OUT/f'PIO_{label}_original_45s.mp4'
    cmd=f'ffmpeg -y -hide_banner -loglevel error -ss {start:.3f} -i "{VIDEO_LOCAL}" -t {CLIP_SECONDS} -c:v libx264 -preset veryfast -crf 20 -an "{out}"'
    rc=os.system(cmd); assert rc==0 and out.exists()
    clip_paths[label]=out; print(f'✓ {label}: {out.name} | inicio={start/60:.2f} min')


## 7. Procesar el clip de alta densidad con YOLO11s

In [ ]:

INPUT_CLIP=clip_paths['alta']; TEMP_OUT=Path('/content/PIO_alta_YOLO_temp.mp4')
FINAL_OUT=VIDEOS_OUT/'PIO_alta_YOLO11s_conteo_45s.mp4'; CSV_OUT=RESULTS_OUT/'NB03_conteo_alta_densidad_por_frame.csv'
cap=cv2.VideoCapture(str(INPUT_CLIP)); assert cap.isOpened()
clip_fps=cap.get(cv2.CAP_PROP_FPS); clip_w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); clip_h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); clip_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
writer=cv2.VideoWriter(str(TEMP_OUT),cv2.VideoWriter_fourcc(*'mp4v'),clip_fps,(clip_w,clip_h))
rows=[]; frame_idx=0; t0=time.time()
while True:
    ok,frame=cap.read()
    if not ok: break
    result=model.predict(source=frame,imgsz=IMGSZ,conf=CONF,iou=IOU,max_det=MAX_DET,device=0,verbose=False)[0]
    n=len(result.boxes) if result.boxes is not None else 0
    mean_conf=float(result.boxes.conf.mean().item()) if n>0 else 0.0
    annotated=result.plot(labels=False,conf=False)
    overlay=annotated.copy(); cv2.rectangle(overlay,(20,20),(520,145),(0,0,0),-1); annotated=cv2.addWeighted(overlay,0.55,annotated,0.45,0)
    cv2.putText(annotated,'AgroUTMACH PIX',(38,58),cv2.FONT_HERSHEY_SIMPLEX,1.0,(255,255,255),2,cv2.LINE_AA)
    cv2.putText(annotated,f'Aves visibles: {n}',(38,98),cv2.FONT_HERSHEY_SIMPLEX,0.9,(255,255,255),2,cv2.LINE_AA)
    cv2.putText(annotated,f'Confianza media: {mean_conf:.2f}',(38,132),cv2.FONT_HERSHEY_SIMPLEX,0.65,(255,255,255),2,cv2.LINE_AA)
    writer.write(annotated)
    rows.append({'frame':frame_idx,'time_seconds':frame_idx/clip_fps,'visible_birds':n,'mean_confidence':mean_conf})
    frame_idx+=1
    if frame_idx % max(1,int(clip_fps*5))==0: print(f'{frame_idx:5d}/{clip_frames:5d} frames | {frame_idx/clip_fps:5.1f}s | elapsed {(time.time()-t0)/60:.1f} min')
cap.release(); writer.release(); pd.DataFrame(rows).to_csv(CSV_OUT,index=False)
cmd=f'ffmpeg -y -hide_banner -loglevel error -i "{TEMP_OUT}" -c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p -an "{FINAL_OUT}"'
rc=os.system(cmd); assert rc==0 and FINAL_OUT.exists()
print('✓ Video YOLO:',FINAL_OUT); print('✓ CSV:',CSV_OUT)


## 8. Resumen del conteo

In [ ]:

df_count=pd.read_csv(CSV_OUT)
summary={'frames_processed':int(len(df_count)),'mean_visible_birds':float(df_count['visible_birds'].mean()),'median_visible_birds':float(df_count['visible_birds'].median()),'min_visible_birds':int(df_count['visible_birds'].min()),'max_visible_birds':int(df_count['visible_birds'].max()),'mean_confidence':float(df_count['mean_confidence'].mean()),'output_video':str(FINAL_OUT)}
print(json.dumps(summary,indent=2,ensure_ascii=False))
with open(RESULTS_OUT/'NB03_resumen_conteo.json','w',encoding='utf-8') as f: json.dump(summary,f,indent=2,ensure_ascii=False)
plt.figure(figsize=(14,5)); plt.plot(df_count['time_seconds'],df_count['visible_birds']); plt.xlabel('Tiempo del clip (s)'); plt.ylabel('Aves visibles detectadas'); plt.title('AgroUTMACH PIX — Conteo por fotograma'); plt.grid(alpha=0.25); plt.tight_layout();
plot_path=RESULTS_OUT/'NB03_conteo_por_tiempo.png'; plt.savefig(plot_path,dpi=180,bbox_inches='tight'); plt.show(); print('✓',plot_path)



## 9. Resultado esperado

```text
04_VIDEOS_FERIA/
├── PIO_alta_original_45s.mp4
├── PIO_media_original_45s.mp4
├── PIO_baja_original_45s.mp4
└── PIO_alta_YOLO11s_conteo_45s.mp4

05_RESULTADOS/NB03_VIDEO/
├── video_metadata.json
├── NB03_muestreo_video.csv
├── NB03_mosaico_12_escenas.png
├── NB03_candidatos_segmentos.json
├── NB03_conteo_alta_densidad_por_frame.csv
├── NB03_conteo_por_tiempo.png
└── NB03_resumen_conteo.json
```

Siguiente etapa: **NB04 — tracking, IDs temporales, trayectorias y movimiento.**


In [ ]:
print('='*72)
print('NB03 FINALIZADO CORRECTAMENTE')
print('Video procesado:', FINAL_OUT)
print(f"Promedio de aves visibles: {summary['mean_visible_birds']:.1f}")
print(f"Máximo de aves visibles: {summary['max_visible_birds']}")
print('='*72)